# Deduplication

## Finding and removing duplicate texts in the British Library collection

The same Shakespeare sonnet appears 47 times in the collection. Three slightly different scans of the Magna Carta. A Victorian railway timetable digitised once in 2008 and again in 2015 under a different catalogue reference.

Deduplication is essential. Duplicate documents waste storage, slow down search, and confuse results — if a researcher searches for "medieval agriculture" and the same passage appears five times in the top ten results, the system looks broken even if each result is technically correct.

But "same" is more nuanced than you might think. Two scans of the same page will differ slightly — one has a smudge, the other has a slightly different OCR interpretation of a faded letter. Are they duplicates? Almost certainly yes. But a character-for-character comparison will say no.

We need exact matching for the easy cases and approximate matching for the hard ones.

## Loading and preparing the data

In [ ]:
import pandas as pd
import hashlib
import re
import html
import unicodedata

df = pd.read_csv("../data/bl_digitised_texts_raw.csv")
print(f"Loaded {len(df)} texts")
df.head()

Before we deduplicate, we should clean the texts first. Duplicate detection on raw text will miss matches where the same underlying content has different HTML tags or invisible characters. Let us apply a basic cleaning pipeline.

In [ ]:
def basic_clean(text):
    """Minimal cleaning for deduplication purposes."""
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = unicodedata.normalize("NFKC", text)
    for char in ["\xa0", "\u200b", "\u200c", "\u200d", "\ufeff"]:
        text = text.replace(char, " " if char == "\xa0" else "")
    # Remove boilerplate headers/footers
    text = re.sub(r"Page \d+ of \d+ -- British Library Digital Collection", "", text)
    text = re.sub(r"--- British Library Digitised Archive --- Page \d+", "", text)
    text = re.sub(r"BL Digital Collections \| Catalogue Ref: [A-Z/0-9]+ \| Page \d+", "", text)
    text = re.sub(r"\[Digitised by the British Library, \d{4}\]", "", text)
    # Remove PII markers
    text = re.sub(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "", text)
    text = re.sub(r"(?:For enquiries contact|Contact:|Digitised by|For access requests, email|or call).*$", "", text, flags=re.MULTILINE)
    # Normalise whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["raw_text"].apply(basic_clean)
print("Cleaning complete.")
print(f"Sample: {df.iloc[0]['clean_text'][:120]}...")

## Exact deduplication with hashing

The simplest form of deduplication: compute a hash of each text and group by hash. Texts with identical hashes are (almost certainly) identical.

A hash function takes an input of any length and produces a fixed-length output — the hash or digest. The same input always produces the same hash. Different inputs *should* produce different hashes, but this is where things get interesting.

In [ ]:
def sha256_hash(text):
    """Compute SHA-256 hash of a text string."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Hash all cleaned texts
df["text_hash"] = df["clean_text"].apply(sha256_hash)

# Find exact duplicates
hash_counts = df["text_hash"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1]

print(f"Unique hashes: {df['text_hash'].nunique()}")
print(f"Duplicate groups: {len(duplicate_hashes)}")
print(f"Total duplicate texts: {(hash_counts > 1).sum()} groups containing {duplicate_hashes.sum()} texts")

In [ ]:
# Look at a duplicate group
if len(duplicate_hashes) > 0:
    example_hash = duplicate_hashes.index[0]
    dupes = df[df["text_hash"] == example_hash]
    print(f"Duplicate group (hash: {example_hash[:16]}...) — {len(dupes)} copies:")
    for _, row in dupes.iterrows():
        print(f"  {row['text_id']} | {row['source_type']} | {row['title'][:50]}")
    print(f"\nText (first 200 chars): {dupes.iloc[0]['clean_text'][:200]}...")

These are exact duplicates: the same underlying text, digitised multiple times under different catalogue entries. Deduplication here is straightforward — keep one copy, discard the rest.

In [ ]:
# Deduplicate: keep first occurrence of each hash
df_exact_deduped = df.drop_duplicates(subset="text_hash", keep="first")
print(f"Before: {len(df)} texts")
print(f"After exact dedup: {len(df_exact_deduped)} texts")
print(f"Removed: {len(df) - len(df_exact_deduped)} exact duplicates")

## The hash collision problem

We used SHA-256, which produces a 256-bit hash. Is that enough? Could two different texts produce the same hash — a collision?

SHA-256 has never had a known collision. The number of possible hashes (2^256) is roughly the number of atoms in the observable universe. You are more likely to win the lottery while being struck by lightning while riding a unicorn.

But shorter hashes are a different story. Let us demonstrate with CRC32, a 32-bit hash commonly used for error detection.

In [ ]:
import binascii

def crc32_hash(text):
    """Compute CRC32 hash (32-bit, only ~4 billion possible values)."""
    return binascii.crc32(text.encode("utf-8")) & 0xFFFFFFFF

# Let's try to find a CRC32 collision by brute force
# With only 2^32 possible values, the birthday paradox means we
# expect a collision after roughly 2^16 = 65,536 random inputs

import random
random.seed(42)

seen_hashes = {}
collision_found = False

for i in range(200_000):
    # Generate a random short text
    text = f"British Library catalogue entry {random.randint(0, 10**12)}-{random.randint(0, 10**12)}"
    h = crc32_hash(text)
    
    if h in seen_hashes and seen_hashes[h] != text:
        print(f"COLLISION FOUND after {i+1} attempts!")
        print(f"  Text A: {seen_hashes[h]!r}")
        print(f"  Text B: {text!r}")
        print(f"  CRC32:  {h}")
        collision_found = True
        break
    
    seen_hashes[h] = text

if not collision_found:
    print("No collision found in 200,000 attempts (unlucky — try increasing the range)")

Two completely different strings, same 32-bit hash. This is why CRC32 is fine for error detection (where you are comparing a known message against itself) but dangerous for deduplication (where you are comparing unknown messages against each other).

| Hash | Bits | Possible values | Collision risk |
|------|------|-----------------|----------------|
| CRC32 | 32 | ~4 billion | Expect collision after ~65K items |
| MD5 | 128 | ~3.4 x 10^38 | Known deliberate collisions |
| SHA-256 | 256 | ~1.2 x 10^77 | No known collision, ever |

For deduplication at scale, SHA-256 is the right choice. The extra computation is negligible; the safety margin is astronomical.

## Near-duplicate detection

Exact hashing catches identical texts, but what about near-duplicates? The same manuscript scanned twice might produce slightly different OCR output. A passage quoted in two different papers is the same content in different contexts.

We need a way to measure *similarity* between texts, not just identity.

### Shingling

The first step is to convert each text into a set of **shingles** — overlapping subsequences of characters or words. Shingles capture local structure: similar texts will share many of the same shingles.

For example, the shingles of "the cat sat" with k=3 words are:
- {"the cat sat"}

With k=2:
- {"the cat", "cat sat"}

In [ ]:
def get_shingles(text, k=3):
    """Generate word-level k-shingles from text."""
    words = text.lower().split()
    if len(words) < k:
        return {tuple(words)}
    return {tuple(words[i:i+k]) for i in range(len(words) - k + 1)}

# Example
text_a = "The feudal system in England following the Norman Conquest of 1066"
text_b = "The feudal system in England after the Norman Conquest of 1066"  # "following" -> "after"

shingles_a = get_shingles(text_a, k=3)
shingles_b = get_shingles(text_b, k=3)

print(f"Text A shingles ({len(shingles_a)}): {list(shingles_a)[:5]}...")
print(f"Text B shingles ({len(shingles_b)}): {list(shingles_b)[:5]}...")
print(f"Shared shingles: {len(shingles_a & shingles_b)}")

### Jaccard similarity

Given two sets of shingles, how similar are they? The **Jaccard similarity** measures this as the size of the intersection divided by the size of the union:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- J = 1.0 means the sets are identical
- J = 0.0 means they share nothing
- J > 0.8 typically indicates near-duplicates

In [ ]:
def jaccard_similarity(set_a, set_b):
    """Compute Jaccard similarity between two sets."""
    if not set_a and not set_b:
        return 1.0
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union

sim = jaccard_similarity(shingles_a, shingles_b)
print(f"Jaccard similarity: {sim:.4f}")
print(f"These texts differ by one word and the similarity is {sim:.1%}")

In [ ]:
# Compare with a completely different text
text_c = "The spinning jenny patented by James Hargreaves in 1770 multiplied productivity eightfold"
shingles_c = get_shingles(text_c, k=3)

sim_ac = jaccard_similarity(shingles_a, shingles_c)
print(f"Similarity between medieval and industrial text: {sim_ac:.4f}")
print(f"As expected — completely different topics, near-zero similarity.")

### Finding near-duplicates in the dataset

Let us compute shingles for a sample of our texts and find pairs that are near-duplicates.

The brute-force approach compares every pair of texts, which is O(n^2). With 2,000 texts, that is roughly 2 million comparisons — feasible for a dataset this size, but we will see a better approach afterwards.

In [ ]:
# Compute shingles for all texts
sample = df_exact_deduped.head(200).copy()  # Use a sample for speed
sample["shingles"] = sample["clean_text"].apply(lambda t: get_shingles(t, k=3))

# Find near-duplicate pairs (Jaccard > 0.5)
near_dupes = []
texts = sample["clean_text"].tolist()
ids = sample["text_id"].tolist()
shingle_list = sample["shingles"].tolist()

for i in range(len(sample)):
    for j in range(i + 1, len(sample)):
        sim = jaccard_similarity(shingle_list[i], shingle_list[j])
        if sim > 0.5:
            near_dupes.append((ids[i], ids[j], sim))

print(f"Checked {len(sample) * (len(sample) - 1) // 2:,} pairs")
print(f"Found {len(near_dupes)} near-duplicate pairs (Jaccard > 0.5)")

# Show some examples
for id_a, id_b, sim in sorted(near_dupes, key=lambda x: -x[2])[:5]:
    print(f"\n  {id_a} <-> {id_b}: Jaccard = {sim:.4f}")
    row_a = sample[sample["text_id"] == id_a].iloc[0]
    row_b = sample[sample["text_id"] == id_b].iloc[0]
    print(f"    A: {row_a['clean_text'][:100]}...")
    print(f"    B: {row_b['clean_text'][:100]}...")

## MinHash: approximate Jaccard at scale

The pairwise comparison approach works for a few hundred texts, but it does not scale. With a million documents, you would need half a trillion comparisons.

**MinHash** is an approximation technique that estimates Jaccard similarity without comparing all pairs. The idea:

1. For each text, compute a *signature* — a small fixed-size array of hash values
2. Two texts with similar Jaccard similarity will have similar signatures
3. Compare signatures (cheap) instead of full shingle sets (expensive)

The signature is built by applying multiple hash functions to each shingle and keeping the minimum hash value for each function. The probability that two signatures agree at any position equals the Jaccard similarity of the original sets. This is a beautiful result from probability theory.

In [ ]:
import numpy as np

class MinHash:
    """MinHash signature generator for approximate Jaccard similarity."""
    
    def __init__(self, num_hashes=128, seed=42):
        self.num_hashes = num_hashes
        # Generate random hash function parameters
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2**31 - 1, size=num_hashes)
        self.b = rng.randint(0, 2**31 - 1, size=num_hashes)
        self.prime = 2**31 - 1  # Mersenne prime
    
    def _hash_shingle(self, shingle):
        """Hash a shingle to an integer."""
        return hash(shingle) & 0x7FFFFFFF
    
    def signature(self, shingles):
        """Compute MinHash signature for a set of shingles."""
        sig = np.full(self.num_hashes, np.inf)
        
        for shingle in shingles:
            h = self._hash_shingle(shingle)
            # Apply each hash function: (a*h + b) mod prime
            values = (self.a * h + self.b) % self.prime
            sig = np.minimum(sig, values)
        
        return sig
    
    def estimated_jaccard(self, sig_a, sig_b):
        """Estimate Jaccard similarity from two signatures."""
        return np.mean(sig_a == sig_b)

mh = MinHash(num_hashes=128)
print("MinHash ready with 128 hash functions.")

In [ ]:
# Compare MinHash estimate with exact Jaccard
sig_a = mh.signature(shingles_a)
sig_b = mh.signature(shingles_b)
sig_c = mh.signature(shingles_c)

print("Exact Jaccard vs MinHash estimate:")
print(f"  A vs B (near-duplicate): exact={jaccard_similarity(shingles_a, shingles_b):.4f}, "
      f"MinHash={mh.estimated_jaccard(sig_a, sig_b):.4f}")
print(f"  A vs C (different):      exact={jaccard_similarity(shingles_a, shingles_c):.4f}, "
      f"MinHash={mh.estimated_jaccard(sig_a, sig_c):.4f}")

The estimates are not exact, but they are in the right ballpark. With 128 hash functions, the standard error of the Jaccard estimate is roughly 1/sqrt(128) = 0.088, or about 9 percentage points. That is good enough to separate near-duplicates (>0.8) from non-duplicates (<0.2) with high confidence.

More hash functions = more accuracy but larger signatures and slower computation. 128 is a practical default.

In [ ]:
# Compute MinHash signatures for all texts in the sample
signatures = []
for shingles in shingle_list:
    signatures.append(mh.signature(shingles))

# Find near-duplicate pairs using MinHash
mh_near_dupes = []
for i in range(len(signatures)):
    for j in range(i + 1, len(signatures)):
        est_sim = mh.estimated_jaccard(signatures[i], signatures[j])
        if est_sim > 0.4:
            mh_near_dupes.append((ids[i], ids[j], est_sim))

print(f"MinHash found {len(mh_near_dupes)} near-duplicate pairs (estimated Jaccard > 0.4)")
for id_a, id_b, sim in sorted(mh_near_dupes, key=lambda x: -x[2])[:5]:
    print(f"  {id_a} <-> {id_b}: estimated Jaccard = {sim:.4f}")

## Quality-based deduplication

When we find duplicates, which copy do we keep? The naive answer is "the first one", but a better approach is to keep the highest-quality version.

Quality heuristics for digitised texts:
- **Length**: longer versions are usually more complete (fewer truncated lines)
- **OCR quality**: fewer suspicious characters (digits where letters should be)
- **Completeness**: fewer missing words, more coherent sentences

In [ ]:
def quality_score(text):
    """Score text quality from 0 to 1. Higher is better."""
    score = 0.0
    
    # Length component (longer is usually better, up to a point)
    length_score = min(len(text) / 500, 1.0)  # max score at 500 chars
    score += 0.4 * length_score
    
    # OCR quality: penalise digits surrounded by letters (likely OCR errors)
    ocr_suspects = len(re.findall(r"[a-zA-Z]\d[a-zA-Z]", text))
    ocr_score = max(0, 1.0 - ocr_suspects * 0.1)
    score += 0.3 * ocr_score
    
    # Completeness: ratio of real words to total tokens
    words = text.split()
    if words:
        real_words = sum(1 for w in words if w.isalpha() and len(w) > 1)
        completeness = real_words / len(words)
    else:
        completeness = 0
    score += 0.3 * completeness
    
    return round(score, 4)

# Score a few examples
good_text = "The feudal system in England following the Norman Conquest of 1066 fundamentally restructured land ownership."
bad_text = "The feuda1 system in Eng1and fo11owing the Norm4n Conquest of 1066"

print(f"Good text quality: {quality_score(good_text)}")
print(f"Bad text quality:  {quality_score(bad_text)}")

In [ ]:
# Score all texts
df["quality"] = df["clean_text"].apply(quality_score)

print("Quality score distribution:")
print(df["quality"].describe().to_string())

In [ ]:
# Quality-based dedup: for each hash group, keep the highest-quality version
df_quality_deduped = df.sort_values("quality", ascending=False).drop_duplicates(
    subset="text_hash", keep="first"
)

print(f"Before: {len(df)} texts")
print(f"After quality-based dedup: {len(df_quality_deduped)} texts")
print(f"Average quality before: {df['quality'].mean():.4f}")
print(f"Average quality after:  {df_quality_deduped['quality'].mean():.4f}")

The average quality improved slightly because we kept the best version of each duplicate group instead of an arbitrary one.

## Putting it all together: a deduplication pipeline

In [ ]:
def dedup_pipeline(df, text_col="clean_text", similarity_threshold=0.8, k=3):
    """Two-stage deduplication pipeline.
    
    Stage 1: Exact dedup via SHA-256 hashing
    Stage 2: Near-duplicate detection via shingling + Jaccard
    """
    result = df.copy()
    
    # Stage 1: Exact dedup
    result["_hash"] = result[text_col].apply(sha256_hash)
    result["_quality"] = result[text_col].apply(quality_score)
    before_exact = len(result)
    result = result.sort_values("_quality", ascending=False).drop_duplicates(
        subset="_hash", keep="first"
    )
    exact_removed = before_exact - len(result)
    
    # Stage 2: Near-duplicate detection on remaining texts
    result["_shingles"] = result[text_col].apply(lambda t: get_shingles(t, k=k))
    
    # Mark near-duplicates for removal
    to_remove = set()
    indices = result.index.tolist()
    shingle_data = result["_shingles"].tolist()
    quality_data = result["_quality"].tolist()
    
    for i in range(len(indices)):
        if indices[i] in to_remove:
            continue
        for j in range(i + 1, len(indices)):
            if indices[j] in to_remove:
                continue
            sim = jaccard_similarity(shingle_data[i], shingle_data[j])
            if sim >= similarity_threshold:
                # Remove the lower-quality one
                if quality_data[i] >= quality_data[j]:
                    to_remove.add(indices[j])
                else:
                    to_remove.add(indices[i])
    
    near_removed = len(to_remove)
    result = result.drop(index=to_remove)
    
    # Clean up temporary columns
    result = result.drop(columns=["_hash", "_quality", "_shingles"])
    
    print(f"Dedup results:")
    print(f"  Input:          {before_exact} texts")
    print(f"  Exact removed:  {exact_removed}")
    print(f"  Near removed:   {near_removed}")
    print(f"  Output:         {len(result)} texts")
    
    return result

In [ ]:
# Run the pipeline on a manageable sample
sample_df = df.head(300).copy()
deduped = dedup_pipeline(sample_df, similarity_threshold=0.8)

---

## Exercises

### Exercise 1: Character-level shingles

Our `get_shingles` function uses word-level shingles. Implement a `get_char_shingles(text, k=8)` function that uses character-level shingles instead (overlapping substrings of length k).

Compare the Jaccard similarity of the word-level and character-level approaches on two near-duplicate texts. Which is more sensitive to small differences? Why?

In [ ]:
# Your code here


### Exercise 2: Tune the similarity threshold

Run the deduplication pipeline on the first 500 texts with thresholds of 0.5, 0.6, 0.7, 0.8, and 0.9. For each threshold, record:
- How many texts survive
- How many near-duplicates are removed

Present the results in a table. What threshold would you recommend for this dataset, and why?

In [ ]:
# Your code here


### Exercise 3: MinHash accuracy

For 50 pairs of texts, compute both the exact Jaccard similarity and the MinHash estimate. Calculate the mean absolute error. Then repeat with 64 and 256 hash functions. How does the number of hash functions affect accuracy?

In [ ]:
# Your code here


---

## Summary

In this notebook we covered three approaches to deduplication:

1. **Exact dedup with hashing** — SHA-256 hashes identify identical texts instantly. CRC32 and other short hashes suffer from collisions and should not be trusted for deduplication.
2. **Near-duplicate detection** — Shingling converts texts to sets of overlapping n-grams. Jaccard similarity measures how much two sets overlap. MinHash approximates Jaccard without exhaustive pairwise comparison.
3. **Quality-based dedup** — When duplicates exist, keep the highest-quality version based on length, OCR quality, and completeness.

The big idea: short hashes are fast but dangerous. SHA-256 has never had a collision in the history of computing. When you are making delete-or-keep decisions about data, use a hash you can trust.

Next up: the texts are clean and deduplicated, but many are too long for effective search. We need to break them into chunks — and where we break them matters enormously.